# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmoud-mos/my-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Ranking/Scoring. This task is structured as a ranking problem because our objective is to prioritize content items based on their likelihood of needing a refresh, enabling efficient allocation of human review capacity.

In [4]:
import pandas as pd

# The ../../ tells Python to go up two folder levels to find the root directory
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

print(f"Total content items queued for priority ranking: {len(df):,}")

Total content items queued for priority ranking: 30,000


## 2. Target or proxy

Binary classification proxy. We define a page as actively decaying—and thus a candidate for refresh—if its `trend_direction` is observed as `"down"`. This proxy serves as our ground-truth label for modeling.

In [5]:
import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print("Target Class Distribution (trend_direction):")
print(df['trend_direction'].value_counts(normalize=True))

Target Class Distribution (trend_direction):
trend_direction
down      0.542067
stable    0.198733
up        0.146267
new       0.074533
flat      0.038400
Name: proportion, dtype: float64


## 3. Success metric

Precision @50 and ROC AUC. We prioritize **Precision @50** to ensure that the top 50 items surfaced for human review are high-impact, minimizing wasted effort. **ROC AUC** is used to evaluate the model's overall ranking performance across the entire dataset.

In [6]:
# Show why Precision@50 matters by comparing it to total decaying pages
decay_count = (df['trend_direction'] == 'down').sum()
print(f"Total decaying pages found: {decay_count:,}")
print(
    f"A review queue of 50 pages handles {(50 / decay_count) * 100:.2f}% of total decaying content at a time.")

Total decaying pages found: 16,262
A review queue of 50 pages handles 0.31% of total decaying content at a time.


## 4. The unit of analysis, as a real dataframe

One row = one content item (`content_hash_id`).

In [7]:
import pandas as pd

# Load the dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Display the head of the dataframe
print("DataFrame Head:")
print(df.head())

# Display the proxy target column
print("\nProxy Target Column ('trend_direction'):")
print(df['trend_direction'].head())

DataFrame Head:
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3  content_331d6c4de07b  client_19581e27de           10.0         0.00   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   
2               LOW  0.00  keyword article  informational      3515.0   
3               LOW  0.00  keyword article     commercial         NaN   
4               LOW  0.00  keyword article  informational      2803.0   

   char_count  ... char_count_tier   ctr  avg_position  engagement_rate  \
0     20457.0  ...     15

## 5. Why ML beats a fixed rule here

ML allows us to dynamically weigh non-linear combinations of signals—such as content age, engagement volume, and CTR gaps—to assess the need for a refresh. Rigid, rule-based thresholds are inherently brittle and fail to adapt to the contextual complexity of content performance over time.

In [8]:
# Show how features vary widely, proving why simple rules are too brittle
print("Feature distributions showing multi-variable complexity:")
print(df[['days_since_last_update', 'impressions_90d', 'clicks_90d']].describe())

Feature distributions showing multi-variable complexity:
       days_since_last_update  impressions_90d    clicks_90d
count            30000.000000     30000.000000  30000.000000
mean                46.098300      5200.366300     16.097333
std                 42.078709     16838.019547     75.076958
min                  1.000000         1.000000      0.000000
25%                 20.000000        81.000000      0.000000
50%                 20.000000       731.000000      1.000000
75%                104.000000      3615.250000      7.000000
max                373.000000    517715.000000   4178.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.